# `partsupp` — profile, choose columns & trim

Trim type: **row**. Choose **4 columns**, **800,000 rows**. Save `partsupp_4c_800000r.csv`.

> Output columns: `c0`…`c3`.

In [3]:
import os
import numpy as np
import pandas as pd

NAME       = "partsupp"
N_ROWS     = 800000
N_COLS     = 5
TRIM_ROWS  = "head"   # head | sample

RAW_PATH   = "partsupp.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ";"
OUT_DELIM  = DELIM
HAS_HEADER = False
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [4]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (800000, 5)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4']


,c0,c1,c2,c3,c4
0,1,2,3325,771.64,", even theodolites. regular, final theodolites..."
1,1,2502,8076,993.49,ven ideas. quickly even packages print. pendin...
2,1,5002,3956,337.09,after the fluffily ironic deposits? blithely s...
3,1,7502,4069,357.84,"al, regular dependencies serve carefully after..."
4,2,3,8895,378.49,nic accounts. final accounts sleep furiously a...


In [ ]:
raw.dtypes

## 2. Profile: cardinality, top-value %, and group skew

In [5]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,200000,0,0.0,25.00,0.00,4.00,4,1.00
c1,10000,0,0.0,1.25,0.01,80.00,80,1.00
c2,9999,0,0.0,1.25,0.01,80.01,112,1.40
c3,99865,0,0.0,12.48,0.00,8.01,21,2.62
c4,799124,0,0.0,99.89,0.00,1.00,2,2.00


## 3. Choose columns (cardinality mix + id/super-key)

In [6]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c0', 'c1', 'c2', 'c3', 'c4']


## 4. Trim to the chosen columns x exact rows

In [7]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (800000, 5)


,c0,c1,c2,c3,c4
0,1,2,3325,771.64,", even theodolites. regular, final theodolites..."
1,1,2502,8076,993.49,ven ideas. quickly even packages print. pendin...
2,1,5002,3956,337.09,after the fluffily ironic deposits? blithely s...
3,1,7502,4069,357.84,"al, regular dependencies serve carefully after..."
4,2,3,8895,378.49,nic accounts. final accounts sleep furiously a...


## 5. Save the trimmed CSV

In [8]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote partsupp_5c_800000r.csv (800000, 5)
reloaded: (800000, 5)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4']


## 6. Check selected cardinality and skew

In [ ]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

In [ ]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof